# Detailed Notebook 2: Interpolation Techniques

**What is interpolation?**
Given a set of known (x, y) data points, interpolation constructs a smooth
function that passes through all of them so you can estimate y at any x
between (and sometimes beyond) your measurements.

**Why do we need it in physics?**
- Tabulated values (material properties, detector efficiencies) are only
  measured at a finite set of points — you often need values in between.
- Expensive simulations (magnetic field maps) are computed on a grid but
  evaluated at arbitrary points during an orbit integration.
- Raw sensor data is discrete; integrating it requires a continuous
  approximation.

**Contents**
1. Quick 1-D linear interpolation with `np.interp`
2. `interp1d` from SciPy — multiple methods (linear, cubic, nearest)
3. `CubicSpline` — the gold standard for smooth 1-D interpolation
4. Polynomial interpolation and the Runge phenomenon
5. 2-D grid interpolation with `RegularGridInterpolator`
6. When to extrapolate (and when not to)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d, CubicSpline, BarycentricInterpolator
from scipy.interpolate import RegularGridInterpolator

---
## Section 1: Quick linear interpolation with `np.interp`

`np.interp` is the simplest tool: it does **piecewise linear** interpolation
and nothing else.  Use it when you just want to look up a value from a table.

### How it works
Between two known points (x0, y0) and (x1, y1), it draws a straight line:

    y(x) = y0 + (y1 - y0) * (x - x0) / (x1 - x0)

It evaluates this formula in whatever segment your query x falls into.

### Function signature
```python
y_query = np.interp(x_query, x_known, y_known)
```

In [ ]:
# ── Example 1a: material property lookup ─────────────────────────────────────
# Suppose a lab measured the resistivity of some alloy at several temperatures.
# We need the resistivity at intermediate temperatures.

T_measured   = np.array([100, 200, 300, 400, 500, 600])   # Kelvin
rho_measured = np.array([1.8, 2.5, 3.3, 4.2, 5.2, 6.3])  # μΩ·cm (made-up values)

# Query at temperatures we did NOT measure
T_query = np.array([150, 250, 350, 450, 550])
rho_query = np.interp(T_query, T_measured, rho_measured)

print('Interpolated resistivities:')
for T, rho in zip(T_query, rho_query):
    print(f'  T = {T} K  →  rho = {rho:.3f} μΩ·cm')

# Plot to see what np.interp actually does (piecewise linear)
T_fine = np.linspace(100, 600, 500)
rho_fine = np.interp(T_fine, T_measured, rho_measured)

plt.figure(figsize=(7, 4))
plt.scatter(T_measured, rho_measured, color='k', s=60, zorder=5, label='measured data')
plt.plot(T_fine, rho_fine, label='np.interp (linear)')
plt.scatter(T_query, rho_query, color='red', s=50, zorder=5, label='interpolated queries')
plt.xlabel('Temperature (K)')
plt.ylabel('Resistivity (μΩ·cm)')
plt.title('np.interp — piecewise linear lookup')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**What to notice:** the interpolated curve has **kinks** at every data point —
this is the cost of piecewise linear interpolation.  At each data point the
slope changes abruptly.  This is fine for tabular lookups but can cause
artefacts when you integrate or differentiate the interpolated function.

---
## Section 2: `interp1d` — multiple interpolation methods

`interp1d` gives you several interpolation strategies in one function call:

| `kind=` | What it does | When to use |
|---------|-------------|-------------|
| `'linear'` | Piecewise straight lines | Fast, sufficient for dense data |
| `'nearest'` | Nearest-neighbour step function | Discrete/categorical data |
| `'quadratic'` | Piecewise degree-2 polynomial | Smoother than linear |
| `'cubic'` | Piecewise degree-3 polynomial | Smooth derivatives, physics |

### How the cubic option works

Between each pair of adjacent data points, it fits a cubic polynomial.
Adjacent cubics are forced to have matching values and first derivatives
at the shared point.  This removes the kinks visible in linear interpolation.

### Function signature

```python
f_interp = interp1d(x_known, y_known, kind='cubic')
y_query  = f_interp(x_query)   # evaluate like a regular function
```

In [ ]:
# ── Example 2a: compare linear vs cubic ─────────────────────────────────────
# Sparse sample from a smooth curve (sin wave with slight perturbation)
x_pts = np.array([0.0, 0.6, 1.1, 1.8, 2.6, 3.2, 4.0])
y_pts = np.sin(x_pts) + 0.05 * np.array([0, 1, -1, 1, -1, 1, 0])

x_fine = np.linspace(x_pts.min(), x_pts.max(), 500)
x_truth = x_fine   # use sin as "truth" for comparison

# Build the two interpolants
# fill_value='extrapolate' is only used here for visualization;
# in practice restrict queries to [x_pts.min(), x_pts.max()].
f_lin  = interp1d(x_pts, y_pts, kind='linear')
f_cub  = interp1d(x_pts, y_pts, kind='cubic')

plt.figure(figsize=(8, 4))
plt.scatter(x_pts, y_pts, color='k', s=60, zorder=6, label='data points')
plt.plot(x_fine, f_lin(x_fine),  label='linear',  linewidth=1.5)
plt.plot(x_fine, f_cub(x_fine),  label='cubic',   linewidth=1.5)
plt.xlabel('x')
plt.ylabel('y')
plt.title('interp1d: linear vs cubic')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**What to notice:**
- The **linear** curve has kinks (discontinuous slope) at every data point.
- The **cubic** curve is smooth everywhere — it looks much more like what
  a physicist would expect from a continuous physical quantity.

The cubic option is almost always preferred in physics unless the data is
very dense or you know the underlying quantity is piecewise linear.

In [ ]:
# ── Example 2b: detector efficiency lookup ───────────────────────────────────
# A detector's efficiency was calibrated at a few photon energies.
# We need to correct measurements at arbitrary energies using this table.

E_cal  = np.array([0.1, 0.3, 0.5, 1.0, 2.0, 5.0, 10.0])  # MeV
eff_cal = np.array([0.45, 0.72, 0.85, 0.90, 0.87, 0.75, 0.55])  # fractional efficiency

eff_interp = interp1d(E_cal, eff_cal, kind='cubic')

# Correct a spectrum measured at these energies
E_meas = np.array([0.2, 0.7, 1.5, 3.0, 7.0])
raw_counts = np.array([120, 340, 450, 280, 95])
corrected = raw_counts / eff_interp(E_meas)

print('Corrected counts:')
for E, raw, corr in zip(E_meas, raw_counts, corrected):
    print(f'  E={E:.1f} MeV  raw={raw}  efficiency={eff_interp(E):.3f}  corrected={corr:.1f}')

---
## Section 3: `CubicSpline` — the gold standard for smooth interpolation

`CubicSpline` is more powerful than `interp1d(..., kind='cubic')` because:
1. You can access **derivatives** of the interpolant analytically.
2. You can specify **boundary conditions** (what the slope is at the ends).
3. It integrates cleanly with the rest of SciPy.

### How a cubic spline works

A cubic spline is a sequence of cubic polynomials, one per interval between
consecutive data points.  The construction imposes four constraints per interval:
1. Match the left data value.
2. Match the right data value.
3. First derivative is continuous at interior points.
4. Second derivative is continuous at interior points.

For N data points this leaves 2 extra degrees of freedom — the boundary conditions.

### Boundary condition options

| `bc_type=` | Meaning |
|-----------|---------|
| `'not-a-knot'` (default) | The third derivative is continuous at x[1] and x[-2] |
| `'natural'` | Second derivative = 0 at the endpoints (gentle curve near edges) |
| `'clamped'` | You specify the first derivative at each endpoint |

### Accessing derivatives

```python
cs = CubicSpline(x, y)
cs(x_query)     # interpolated values (the spline itself)
cs(x_query, 1)  # first derivative  dy/dx
cs(x_query, 2)  # second derivative d²y/dx²
```

In [ ]:
# ── Example 3a: basic spline and its derivative ──────────────────────────────
x_pts = np.array([0.0, 0.6, 1.1, 1.8, 2.6, 3.2, 4.0])
y_pts = np.sin(x_pts) + 0.05 * np.array([0, 1, -1, 1, -1, 1, 0])

cs = CubicSpline(x_pts, y_pts, bc_type='natural')

x_fine = np.linspace(0, 4, 500)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(x_pts, y_pts, color='k', zorder=6, label='data')
axes[0].plot(x_fine, cs(x_fine), label='CubicSpline')
axes[0].set_xlabel('x');  axes[0].set_ylabel('y')
axes[0].set_title('Spline (value)');  axes[0].legend();  axes[0].grid(True, alpha=0.3)

axes[1].plot(x_fine, cs(x_fine, 1), color='green', label='1st derivative')
axes[1].plot(x_fine, cs(x_fine, 2), color='orange', label='2nd derivative')
axes[1].set_xlabel('x')
axes[1].set_title('Spline derivatives');  axes[1].legend();  axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Why derivatives matter in physics:**
- Velocity is the derivative of position → if you have position tabulated over time,
  `cs(t, 1)` gives velocity at any time.
- The second derivative lets you check whether the interpolated curve is physically
  reasonable (e.g., no unphysical oscillations in a potential).

In [ ]:
# ── Example 3b: integrate the spline ────────────────────────────────────────
# CubicSpline objects also have a built-in .integrate() method.
# Useful for: computing impulse from a force-time curve, or
#             total energy deposited from a power vs time measurement.

# Example: integrate a tabulated force curve F(t) over [0, 4] to get impulse J
t_pts = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0])
F_pts = np.array([0.0, 3.2, 5.8, 4.1, 2.3, 1.0, 0.0])  # Newtons

cs_force = CubicSpline(t_pts, F_pts)
impulse = cs_force.integrate(0, 4)
print(f'Impulse J = ∫F dt over [0,4] = {impulse:.4f} N·s')

t_fine = np.linspace(0, 4, 300)
plt.figure(figsize=(7, 4))
plt.fill_between(t_fine, cs_force(t_fine), alpha=0.2, label='area = impulse')
plt.scatter(t_pts, F_pts, color='k', zorder=6)
plt.plot(t_fine, cs_force(t_fine), 'b', label='CubicSpline of F(t)')
plt.xlabel('time (s)')
plt.ylabel('Force (N)')
plt.title('Integrating a force curve with CubicSpline')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## Section 4: Polynomial interpolation and the Runge phenomenon

`BarycentricInterpolator` fits a **single polynomial of degree N-1** through all
N data points.

### When is this a problem?

A high-degree polynomial through equally spaced points **oscillates wildly near
the edges** of the data range — this is the **Runge phenomenon**.

Example: the function 1/(1+25x²) sampled at equally spaced points is well-known
to show this behaviour.

### When is polynomial interpolation fine?

If you use **Chebyshev nodes** (points clustered near the edges) instead of equally
spaced points, the oscillation disappears.  The class notes use `chebinterpolate`
for exactly this reason.

In [ ]:
# ── Example 4: Runge phenomenon demonstration ────────────────────────────────
def runge(x):
    return 1.0 / (1 + 25 * x**2)

# Sample at 11 equally-spaced points
N = 11
x_eq   = np.linspace(-1, 1, N)
y_eq   = runge(x_eq)

# Sample at 11 Chebyshev nodes
k      = np.arange(1, N+1)
x_cheb = np.cos((2*k - 1) * np.pi / (2*N))   # Chebyshev nodes on [-1,1]
y_cheb = runge(x_cheb)

# Build polynomial interpolants
bi_eq   = BarycentricInterpolator(x_eq,   y_eq)
bi_cheb = BarycentricInterpolator(x_cheb, y_cheb)

x_fine = np.linspace(-1, 1, 500)

plt.figure(figsize=(10, 5))
plt.plot(x_fine, runge(x_fine),       'k',   label='true function 1/(1+25x²)', linewidth=1.5)
plt.plot(x_fine, bi_eq(x_fine),   '--r',  label='poly interp: equally spaced (RUNGE!)')
plt.plot(x_fine, bi_cheb(x_fine), '--g',  label='poly interp: Chebyshev nodes')
plt.scatter(x_eq,   y_eq,   color='red',   s=30, zorder=5)
plt.scatter(x_cheb, y_cheb, color='green', s=30, zorder=5)
plt.ylim(-0.5, 1.5)
plt.xlabel('x');  plt.ylabel('y')
plt.title('Runge phenomenon: equally-spaced vs Chebyshev nodes')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**What to notice:**
- The red dashed curve (equally-spaced nodes) oscillates massively near x = ±1
  even though the true function is perfectly smooth there.
- The green dashed curve (Chebyshev nodes) matches the true function very well.

**Takeaway:** if you must use polynomial interpolation, use Chebyshev nodes.
In practice, `CubicSpline` (piecewise cubic) is usually a safer choice for
arbitrary data.

---
## Section 5: 2-D grid interpolation with `RegularGridInterpolator`

Many physics problems involve field quantities defined on a 2-D grid:
magnetic field maps, potential maps, temperature distributions.

`RegularGridInterpolator` interpolates on a rectangular grid and returns values
at arbitrary (x, y) points inside the grid.

### Function signature

```python
from scipy.interpolate import RegularGridInterpolator

# x_grid and y_grid are 1-D arrays of grid coordinates
# Z is a 2-D array with shape (len(x_grid), len(y_grid))
interp2 = RegularGridInterpolator((x_grid, y_grid), Z, method='linear')

# Query at arbitrary points (can be many at once)
pts = np.array([[x1, y1], [x2, y2], ...])
values = interp2(pts)
```

In [ ]:
# ── Example 5a: interpolate a 2-D Gaussian field ────────────────────────────
# Imagine this is a magnetic field magnitude measured on a coarse grid.

x_grid = np.linspace(-3, 3, 20)
y_grid = np.linspace(-3, 3, 18)
Xg, Yg = np.meshgrid(x_grid, y_grid, indexing='ij')
Z_field = np.exp(-(Xg**2 + Yg**2) / 2.0)   # Gaussian shape

interp2 = RegularGridInterpolator((x_grid, y_grid), Z_field, method='linear')

# Query at a few specific points not on the grid
query_pts = np.array([[0.0, 0.0],
                      [1.5, 0.5],
                      [-2.0, 1.0],
                      [2.5, -2.5]])

values = interp2(query_pts)
for pt, val in zip(query_pts, values):
    exact = np.exp(-(pt[0]**2 + pt[1]**2) / 2.0)
    print(f'  ({pt[0]:5.1f}, {pt[1]:5.1f}):  interp = {val:.6f},  exact = {exact:.6f},  error = {abs(val-exact):.2e}')

In [ ]:
# ── Example 5b: orbit through a field map ───────────────────────────────────
# A common workflow: simulate a particle trajectory through a field defined
# on a grid.  At each trajectory point we call the interpolant to get the field.

from scipy.integrate import solve_ivp

# Suppose the 2-D field is B_z(x,y) and the particle moves in the xy-plane.
# For simplicity, just trace a trajectory and sample the field along it.

# (Reuse interp2 from above)
def trajectory_ode(t, state):
    x, y, vx, vy = state
    # Clamp to grid to avoid extrapolation
    xc = np.clip(x, x_grid.min()+0.01, x_grid.max()-0.01)
    yc = np.clip(y, y_grid.min()+0.01, y_grid.max()-0.01)
    Bz = float(interp2([[xc, yc]]))
    q_over_m = 1.0
    # Lorentz force: a = q/m * v × B  (z component of B, motion in xy)
    ax =  q_over_m * vy * Bz
    ay = -q_over_m * vx * Bz
    return [vx, vy, ax, ay]

sol_orbit = solve_ivp(trajectory_ode, [0, 20], [0.5, 0.0, 0.0, 1.0],
                      t_eval=np.linspace(0, 20, 1000), max_step=0.05)

plt.figure(figsize=(6, 6))
plt.contourf(x_grid, y_grid, Z_field.T, levels=20, cmap='Blues', alpha=0.5)
plt.colorbar(label='B_z field')
plt.plot(sol_orbit.y[0], sol_orbit.y[1], 'r', linewidth=1.5, label='particle orbit')
plt.scatter([0.5], [0.0], color='green', zorder=5, s=60, label='start')
plt.xlabel('x');  plt.ylabel('y')
plt.title('Particle orbit through an interpolated field map')
plt.legend()
plt.axis('equal')
plt.show()

---
## Section 6: Extrapolation — when to do it and when to avoid it

**Interpolation**: estimating values *within* the range of your data.
**Extrapolation**: estimating values *outside* the range of your data.

Extrapolation is dangerous because:
- Any polynomial or spline behaves unpredictably beyond the last data point.
- The physical system may have a different regime outside your measurements
  (phase transitions, saturation, new physics).

### Example: see how extrapolation diverges

In [ ]:
# ── Example 6: extrapolation divergence ─────────────────────────────────────
x_sparse = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
y_sparse  = np.sin(x_sparse)

cs_extrap = CubicSpline(x_sparse, y_sparse)

x_inside  = np.linspace(0, 4, 300)     # interpolation region
x_outside = np.linspace(-1, 6, 300)    # includes extrapolation

plt.figure(figsize=(9, 4))
plt.plot(x_outside, np.sin(x_outside),           'k',   label='true sin(x)')
plt.plot(x_inside,  cs_extrap(x_inside),  'b',   label='spline (interpolation)', linewidth=2)
plt.plot(x_outside, cs_extrap(x_outside), 'r--', label='spline (extrapolation)', linewidth=1.5)
plt.scatter(x_sparse, y_sparse, color='k', zorder=6, s=60, label='data')
plt.axvspan(-1, 0, alpha=0.1, color='red', label='extrapolation region')
plt.axvspan(4, 6,  alpha=0.1, color='red')
plt.ylim(-3, 3)
plt.xlabel('x');  plt.ylabel('y')
plt.title('Extrapolation error grows rapidly outside data range')
plt.legend(loc='lower left')
plt.grid(True, alpha=0.3)
plt.show()

**What to notice:**
Inside [0, 4] the spline tracks sin(x) well.  Outside, it diverges rapidly.

**Best practice:** always check that your query points lie within the range of
your data.  If they do not, either acquire more data, or use a physically
motivated model to extrapolate rather than relying on pure interpolation.

---
## Summary: which interpolation tool to use?

| Tool | Best for |
|------|---------|
| `np.interp` | Simple tabular lookup, no dependencies needed |
| `interp1d(..., kind='cubic')` | Quick smooth 1-D interpolation |
| `CubicSpline` | Need derivatives, integrals, or boundary conditions |
| `BarycentricInterpolator` | Polynomial with Chebyshev nodes |
| `RegularGridInterpolator` | 2-D/3-D field data on a regular grid |